
##Import the required library

In [0]:
import requests
import json
from pyspark.sql.functions import *
from pyspark.sql.types import * 


###Reading the data from Bronze Table

In [0]:
bronze_df = spark.table('cricket_api.cricket.bronze_current_matches')

raw_json = bronze_df.select('raw_json').collect()[0]['raw_json']
api_data = json.loads(raw_json)
matches = api_data.get("data",[])

print("Total Matches found : ",len(matches))

print(matches[0] if len(matches)>0 else "No Matches Found")


#### Extract Only Useful fields

In [0]:
silver_rows = []

for match in matches :
    teams = match.get("teams",[])
    score = match.get("score",[])
    #print(teams,score)

    team_1 = teams[0] if len(teams)>0 else None
    team_2 = teams[1] if len(teams)>1 else None

    score_1 = None
    score_2 = None

    #formatting it in real score display 135/6 in 20 Overs
    if len(score)>0:
        s1 = score[0]
        score_1 = f"{s1.get('r',0)}/{s1.get('w',0)} in {s1.get('o',0)} overs"
    if len(score)>1:
        s2 = score[1]
        score_2 = f"{s2.get('r',0)}/{s2.get('w',0)} in {s2.get('o',0)} overs"

    silver_rows.append({
        "match_id" : match.get("id"),
        "match_name" : match.get("name"),
        "match_type" : match.get("matchType"),
        "match_result" : match.get("status"),
        "venue" : match.get("venue"),
        "match_date" : match.get("date"),
        "date_time_gmt" : match.get("dateTimeGMT"),
        "team_1" : team_1,
        "team_2" : team_2,
        "score_1" : score_1,
        "score_2" : score_2,
        "match_started" : match.get("matchStarted"),
        "match_ended" : match.get("matchEnded")
    }
    )
print("Silver rows prepared ", len(silver_rows))


    

In [0]:
silver_schema = StructType([
    StructField("match_id", StringType(), True),
    StructField("match_name", StringType(), True),
    StructField("match_type", StringType(), True),
    StructField("match_result", StringType(), True),
    StructField("venue", StringType(), True),
    StructField("match_date", StringType(), True),
    StructField("date_time_gmt", StringType(), True),
    StructField("team_1", StringType(), True),
    StructField("team_2", StringType(), True),
    StructField("score_1", StringType(), True),
    StructField("score_2", StringType(), True),
    StructField("match_started", BooleanType(), True),
    StructField("match_ended", BooleanType(), True)
])

In [0]:
silver_df = spark.createDataFrame(silver_rows,schema=silver_schema)\
    .withColumn("match_date",to_date(col("match_date")))\
        .withColumn("date_time_gmt",to_timestamp(col("date_time_gmt")))\
            .withColumn("loaded_at",current_timestamp())

In [0]:
silver_df.display()

In [0]:
silver_df.write.format("delta")\
    .mode("overwrite")\
        .option("overwriteSchema",True)\
            .saveAsTable("cricket_api.cricket.silver_current_matches")

In [0]:
%sql

Select * from cricket_api.cricket.silver_current_matches